# RAG on Azure — Day 14: The Evaluation Harness 

**The problem:** since Day 8 we've shipped six retrieval upgrades — hybrid
search, reranking, parent-document, sentence-window, contextual chunks,
metadata filters — and the only evidence any of them helps is *"the demo
looked better."* That's not engineering, that's vibes. Nobody would ship a
model without a test set; RAG pipelines deserve the same.


**The four pieces:**

1. **An eval set** — 20 questions over the combined Day 12 + Day 13 corpora,
   each with ground-truth source documents and a reference answer
   (`evalset_day14.json`, versioned in the repo like code).
2. **Pipeline variants** — the systems under test:
   `baseline` (raw chunks, vector-only) → `contextual` (Day 12) →
   `ctx+filter` (Day 13's `isLatest`) → `hybrid+ctx+filter` (Day 8 + 12 + 13).
3. **Metrics** — retrieval: hit@k, precision@k, MRR (pure math, no LLM);
   generation: faithfulness and answer correctness (LLM-as-judge).
4. **A leaderboard + regression file** — aggregate scores per variant, a
   chart, and timestamped JSON results so future changes can be diffed.


## 1. Setup & configuration

Same pattern as Days 1–13. We reuse **both** earlier corpora — no new
documents today; the eval set is the new data.

In [ ]:
import os, json, re, time, hashlib
from collections import defaultdict

try:
    from common.azure_clients import load_config, get_openai_client, get_search_client
except ImportError:
    from openai import AzureOpenAI
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    def load_config(path="config.json"):
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
        return {
            "openai_endpoint":      os.environ["AZURE_OPENAI_ENDPOINT"],
            "openai_key":           os.environ["AZURE_OPENAI_KEY"],
            "search_endpoint":      os.environ["AZURE_SEARCH_ENDPOINT"],
            "search_key":           os.environ["AZURE_SEARCH_KEY"],
            "chat_deployment":      os.environ.get("AZURE_CHAT_DEPLOYMENT", "gpt-4o-mini"),
            "embedding_deployment": os.environ.get("AZURE_EMBED_DEPLOYMENT", "text-embedding-3-small"),
        }

    def get_openai_client(config):
        return AzureOpenAI(azure_endpoint=config["openai_endpoint"],
                           api_key=config["openai_key"], api_version="2024-10-21")

    def get_search_client(config, index_name):
        return SearchClient(endpoint=config["search_endpoint"], index_name=index_name,
                            credential=AzureKeyCredential(config["search_key"]))

config = load_config()
openai = get_openai_client(config)

INDEX_RAW = "rag-eval-day14-raw"   # raw chunks
INDEX_CTX = "rag-eval-day14-ctx"   # contextualized chunks (Day 12 treatment)

# Reuse the Day 12 + Day 13 corpora (10 PDFs total)
DATA_DIRS = [
    "../day-12-contextual-retrieval/data-day12",
    "../day-13-metadata-retrieval/data-day13",
]

print("Config loaded. Chat:", config["chat_deployment"],
      "| Embeddings:", config["embedding_deployment"])

## 2. Load the eval set

20 questions, each with: the question, the **source document(s)** a correct
retrieval must surface (doc-level ground truth), a **reference answer** for
generation scoring, and a **tag** — `factual` (answer lives in one current
document) or `history` (answer needs superseded versions). The tag exists to
catch a subtle failure you'll see in section 8.

Treat this file like code: version it, review changes to it, grow it as users
surface new question patterns.

In [ ]:
with open("evalset_day14.json") as f:
    EVAL = json.load(f)["items"]

print(f"{len(EVAL)} eval questions "
      f"({sum(e['tag']=='factual' for e in EVAL)} factual, "
      f"{sum(e['tag']=='history' for e in EVAL)} history)\n")
for e in EVAL[:3]:
    print(f"  [{e['id']}] {e['question']}")
print("  ...")

## 3. Ingest the combined corpus

One ingestion path handles both corpus styles: Day 13 documents carry a
Document Control block (parsed into metadata + version grouping → `isLatest`);
Day 12 quarterly reports don't, so they get sensible defaults
(`docType=Report`, `isLatest=true` — each report is the only version of its
period). Chunking mirrors the per-section logic from those days.

In [ ]:
from pypdf import PdfReader

META_LABELS = ["Document Type", "Version", "Effective Date",
               "Author", "Department", "Status"]
REPORT_HEADINGS = ["Financial Highlights", "Revenue Performance",
                   "Margins and Costs", "Outlook", "Principal Risks"]

def parse_control_block(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return {l: lines[i+1] for i, l in enumerate(lines)
            if l in META_LABELS and i + 1 < len(lines)}

def split_report(full_text):
    pattern = "(" + "|".join(re.escape(h) for h in REPORT_HEADINGS) + ")"
    parts = re.split(pattern, full_text)
    return [(parts[i].strip(), parts[i+1].strip())
            for i in range(1, len(parts) - 1, 2)]

def split_corporate(text):
    body = text.split("Status", 1)[-1].split("\n", 1)[-1]
    secs, h, cur = [], None, []
    for line in body.splitlines():
        if line and len(line) < 60 and not line.endswith(".") and line[0].isupper() \
           and not line[0].isdigit() and len(line.split()) <= 6:
            if h and cur:
                secs.append((h, " ".join(cur).strip()))
            h, cur = line.strip(), []
        else:
            cur.append(line.strip())
    if h and cur:
        secs.append((h, " ".join(cur).strip()))
    return [(a, b) for a, b in secs if len(b) > 40]

documents, doc_meta = {}, {}
for d in DATA_DIRS:
    for fname in sorted(os.listdir(d)):
        if not fname.endswith(".pdf"):
            continue
        full = "\n".join(p.extract_text() for p in PdfReader(os.path.join(d, fname)).pages)
        documents[fname] = full
        meta = parse_control_block(full)
        title = full.splitlines()[0].strip()
        doc_meta[fname] = {
            "title": title,
            "docType":  meta.get("Document Type", "Report"),
            "version":  float(meta.get("Version", 1.0)),
            "hasMeta":  bool(meta),
        }

# isLatest: version-grouped for control-block docs; reports stand alone
families = defaultdict(list)
for f, m in doc_meta.items():
    key = m["title"] if m["hasMeta"] else f
    families[key].append(f)
for members in families.values():
    members.sort(key=lambda f: doc_meta[f]["version"])
    for f in members:
        doc_meta[f]["isLatest"] = (f == members[-1])

chunks = []
for fname, full in documents.items():
    sections = split_corporate(full) if doc_meta[fname]["hasMeta"] else split_report(full)
    for heading, body_text in sections:
        chunks.append({
            "id":       hashlib.md5(f"{fname}-{heading}".encode()).hexdigest()[:12],
            "source":   fname,
            "section":  heading,
            "content":  f"{doc_meta[fname]['title']} — {heading}\n{body_text}",
            "docType":  doc_meta[fname]["docType"],
            "isLatest": doc_meta[fname]["isLatest"],
        })

print(f"{len(documents)} documents -> {len(chunks)} chunks")
for f, m in doc_meta.items():
    print(f"   {f:34s} latest={m['isLatest']}")

## 4. Build the two indexes

Two physical indexes cover all four variants: `raw` (baseline) and `ctx`
(Day 12's situating treatment). Filters and hybrid mode are query-time
choices, so they don't need their own indexes — that's the point of putting
metadata in the schema.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery

EMBED_DIMS = 1536

def embed(texts, batch=16):
    out = []
    for i in range(0, len(texts), batch):
        resp = openai.embeddings.create(model=config["embedding_deployment"],
                                        input=texts[i:i+batch])
        out.extend(d.embedding for d in resp.data)
    return out

def build_index(name):
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source",   type=SearchFieldDataType.String,  filterable=True),
        SimpleField(name="section",  type=SearchFieldDataType.String,  filterable=True),
        SimpleField(name="docType",  type=SearchFieldDataType.String,  filterable=True),
        SimpleField(name="isLatest", type=SearchFieldDataType.Boolean, filterable=True),
        SearchField(name="contentVector",
                    type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                    searchable=True, vector_search_dimensions=EMBED_DIMS,
                    vector_search_profile_name="vp"),
    ]
    vs = VectorSearch(algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
                      profiles=[VectorSearchProfile(name="vp",
                                                    algorithm_configuration_name="hnsw")])
    ic = SearchIndexClient(config["search_endpoint"],
                           AzureKeyCredential(config["search_key"]))
    try:
        ic.delete_index(name)
    except Exception:
        pass
    ic.create_index(SearchIndex(name=name, fields=fields, vector_search=vs))

def upload(name, records):
    vectors = embed([r["content"] for r in records])
    get_search_client(config, name).upload_documents(
        documents=[{**r, "contentVector": v} for r, v in zip(records, vectors)])
    time.sleep(2)
    print(f"'{name}': {len(records)} chunks indexed.")

# --- raw index ---
build_index(INDEX_RAW)
upload(INDEX_RAW, chunks)

In [ ]:
# --- contextualized index (Day 12 treatment: one situating call per chunk) ---
CONTEXT_PROMPT = """<document>
{document}
</document>

Here is a chunk taken from the document above:

<chunk>
{chunk}
</chunk>

Write 1-2 short sentences situating this chunk within the overall document so
the chunk can be understood on its own. Always name the organization or policy
and the version/period. Respond with ONLY the situating sentences."""

def situate(doc_text, chunk_text):
    resp = openai.chat.completions.create(
        model=config["chat_deployment"], temperature=0, max_tokens=120,
        messages=[{"role": "user", "content":
                   CONTEXT_PROMPT.format(document=doc_text, chunk=chunk_text)}])
    return resp.choices[0].message.content.strip()

ctx_chunks = []
for c in chunks:
    note = situate(documents[c["source"]], c["content"])
    ctx_chunks.append({**c, "content": f"{note}\n\n{c['content']}"})
print(f"Situated {len(ctx_chunks)} chunks.")

build_index(INDEX_CTX)
upload(INDEX_CTX, ctx_chunks)

## 5. The systems under test

A variant is just a configuration: which index, hybrid on/off, filter on/off.
The harness doesn't care how a variant works internally — it only needs
`retrieve(question) -> hits` and `answer(question, hits) -> text`. That
interface is what makes the harness reusable for Days 15–30.

In [ ]:
VARIANTS = {
    "baseline":          {"index": INDEX_RAW, "hybrid": False, "filter": None},
    "contextual":        {"index": INDEX_CTX, "hybrid": False, "filter": None},
    "ctx+filter":        {"index": INDEX_CTX, "hybrid": False, "filter": "isLatest eq true"},
    "hybrid+ctx+filter": {"index": INDEX_CTX, "hybrid": True,  "filter": "isLatest eq true"},
}

K = 3  # retrieval depth under test

def retrieve(variant, question, k=K):
    v = VARIANTS[variant]
    qv = embed([question])[0]
    results = get_search_client(config, v["index"]).search(
        search_text=question if v["hybrid"] else None,   # hybrid = BM25 + vector
        vector_queries=[VectorizedQuery(vector=qv, k_nearest_neighbors=k,
                                        fields="contentVector")],
        filter=v["filter"],
        select=["source", "section", "content"], top=k,
    )
    return list(results)

def answer(question, hits):
    context = "\n\n---\n\n".join(
        f"[{h['source']} / {h['section']}]\n{h['content']}" for h in hits)
    resp = openai.chat.completions.create(
        model=config["chat_deployment"], temperature=0,
        messages=[
            {"role": "system",
             "content": "Answer ONLY from the provided context. Be concise. "
                        "If the context is insufficient, say so."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ])
    return resp.choices[0].message.content.strip()

## 6. Metrics

**Retrieval (deterministic, free, run constantly):**
- **hit@k** — did *any* expected source appear in the top-k? (doc-level recall)
- **precision@k** — what fraction of retrieved chunks came from expected sources?
- **MRR** — reciprocal rank of the first correct chunk (1.0 = top hit correct).

**Generation (LLM-as-judge, costs money, run on changes):**
- **faithfulness** — is every claim in the answer supported by the retrieved
  context? Catches hallucination *given* the context.
- **correctness** — does the answer agree with the reference answer? Catches
  wrong-document answers that are perfectly faithful to the wrong context.

That last distinction is the one to internalize: **an answer can be 100%
faithful and 100% wrong** — faithfully grounded in a superseded policy. You
need both metrics to see it.

In [ ]:
def retrieval_metrics(hits, expected_sources):
    sources = [h["source"] for h in hits]
    hit = float(any(s in expected_sources for s in sources))
    precision = sum(s in expected_sources for s in sources) / max(len(sources), 1)
    mrr = 0.0
    for rank, s in enumerate(sources, start=1):
        if s in expected_sources:
            mrr = 1.0 / rank
            break
    return {"hit@k": hit, "precision@k": precision, "mrr": mrr}

JUDGE_FAITHFULNESS = """You are a strict evaluator. Given a CONTEXT and an ANSWER,
decide whether every factual claim in the ANSWER is directly supported by the CONTEXT.
Respond with ONLY a JSON object: {{"score": <0.0 to 1.0>, "reason": "<one sentence>"}}
Score 1.0 = fully supported, 0.5 = partially, 0.0 = unsupported or contradicted.

CONTEXT:
{context}

ANSWER:
{answer}"""

JUDGE_CORRECTNESS = """You are a strict evaluator. Compare the CANDIDATE answer to the
REFERENCE answer for the QUESTION. Judge factual agreement, not wording.
Respond with ONLY a JSON object: {{"score": <0.0 to 1.0>, "reason": "<one sentence>"}}
Score 1.0 = same facts, 0.5 = partially correct or incomplete, 0.0 = wrong or unsupported.

QUESTION: {question}
REFERENCE: {reference}
CANDIDATE: {candidate}"""

def judge(prompt):
    resp = openai.chat.completions.create(
        model=config["chat_deployment"], temperature=0, max_tokens=150,
        messages=[{"role": "user", "content": prompt}])
    text = resp.choices[0].message.content.strip()
    try:
        return json.loads(re.search(r"\{.*\}", text, re.S).group())
    except Exception:
        return {"score": 0.0, "reason": f"unparseable judge output: {text[:80]}"}

## 7. Run the harness

20 questions × 4 variants. Retrieval metrics are computed for every run;
generation judging runs on every (question, variant) pair too — at this corpus
size that's cheap. At production scale you'd run retrieval metrics on every
commit and judged metrics nightly.

In [ ]:
import pandas as pd

rows = []
for variant in VARIANTS:
    print(f"Running variant: {variant}")
    for e in EVAL:
        hits = retrieve(variant, e["question"])
        rmet = retrieval_metrics(hits, e["expected_sources"])
        ans  = answer(e["question"], hits)
        context = "\n".join(h["content"] for h in hits)
        faith = judge(JUDGE_FAITHFULNESS.format(context=context, answer=ans))
        corr  = judge(JUDGE_CORRECTNESS.format(
            question=e["question"], reference=e["reference_answer"], candidate=ans))
        rows.append({
            "variant": variant, "qid": e["id"], "tag": e["tag"], **rmet,
            "faithfulness": faith["score"], "correctness": corr["score"],
            "answer": ans, "correctness_reason": corr["reason"],
        })

df = pd.DataFrame(rows)
print(f"\n{len(df)} evaluations complete.")

### The leaderboard

In [ ]:
METRICS = ["hit@k", "precision@k", "mrr", "faithfulness", "correctness"]

leaderboard = df.groupby("variant")[METRICS].mean().round(3)
leaderboard = leaderboard.reindex(list(VARIANTS))   # keep build order
print(leaderboard.to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(11, 5.5), dpi=120)
x, width = np.arange(len(METRICS)), 0.2
colors = ["#9CA3AF", "#60A5FA", "#2563EB", "#7C3AED"]
for i, variant in enumerate(leaderboard.index):
    ax.bar(x + (i - 1.5) * width, leaderboard.loc[variant, METRICS],
           width, label=variant, color=colors[i])
ax.set_xticks(x); ax.set_xticklabels(METRICS)
ax.set_ylim(0, 1.05); ax.set_ylabel("score (mean over 20 questions)")
ax.set_title("Day 14 — RAG variant leaderboard")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/day14_leaderboard.png", bbox_inches="tight")
plt.show()

## 8. The result that justifies the whole harness

Aggregate numbers hide trade-offs. Split the leaderboard by question tag and a
story appears that no demo would have shown you:

In [ ]:
by_tag = (df.groupby(["variant", "tag"])[["hit@k", "correctness"]]
            .mean().round(3).unstack("tag"))
print(by_tag.to_string())
print()
hist = df[(df.tag == "history")]
for v in VARIANTS:
    row = hist[hist.variant == v]
    print(f"history question under '{v}': hit@k={row['hit@k'].mean():.2f}, "
          f"correctness={row['correctness'].mean():.2f}")

Expected pattern: the `isLatest` filter **lifts** factual/current
questions (the version trap from Day 13 disappears) and **breaks** the history
question — *"how did the policy change across versions?"* can't be answered
when superseded versions are filtered out of retrieval. The filter isn't wrong;
**applying it unconditionally is.** That insight points directly at Days 16–17
(query understanding and routing: choose filters per query intent).

This is what an eval harness is *for*: not a single score, but visibility into
which technique helps **which class of question** — so "improvements" stop
silently breaking things elsewhere.

## 9. Persist for regression tracking

Every run writes a timestamped JSON. From now on, any change — new chunking,
new embedding model, new prompt — gets a before/after diff instead of an
opinion.

In [ ]:
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
payload = {
    "run_id": stamp,
    "k": K,
    "variants": {v: VARIANTS[v] for v in VARIANTS},
    "leaderboard": leaderboard.reset_index().to_dict(orient="records"),
    "per_question": df.drop(columns=["answer"]).to_dict(orient="records"),
}
out = f"results/day14_results_{stamp}.json"
with open(out, "w") as f:
    json.dump(payload, f, indent=2)
print("wrote", out)

## 10. Takeaways

- **Faithful ≠ correct.** An answer grounded perfectly in the wrong (e.g.
  superseded) document scores 1.0 on faithfulness and 0.0 on correctness.
  Measuring only one of the two is how confident failures ship.
- **Aggregate scores hide regressions.** The `isLatest` filter wins overall
  and loses on history questions. Tag your eval set and always read the
  per-tag split.
- **Retrieval metrics are free; judged metrics are not.** hit@k / precision /
  MRR are pure math — run them on every change. LLM-judged faithfulness and
  correctness cost tokens — run them on candidate changes and nightly.
- **The eval set is an asset, not a script.** 20 questions is a starting
  point; every production incident should become a new eval question. Version
  it, review it, grow it.
- **The harness outlives every technique.** Days 15–30 (compression, routing,
  CRAG, Self-RAG, agentic retrieval) will each be judged by this exact
  yardstick — that's why the variant interface is just `retrieve()` + `answer()`.